## Managed Memory for Databricks Agents

### Installing Utilities and Libraries

In [ ]:
%pip install databricks-sdk==0.49.0 openai-agents==0.22.0 databricks-openai==0.17.1

### Restart your Python Environment

In [ ]:
dbutils.library.restartPython()

### Generate the OAuth Token for Memory Creation Purpose

In [ ]:
import requests

workspace_host = "YOUR-DATABRICKS-HOSTNAME-GOES-HERE"

client_id = "YOUR-SVC-PRINCIPAL-CLIENT-GOES-HERE"
client_secret = "YOUR-SVC-PRINCIPAL-SECRET-GOES-HERE"

response = requests.post(
    f"{workspace_host}/oidc/v1/token",
    auth=(client_id, client_secret),
    data={
        "grant_type": "client_credentials",
        "scope": "all-apis"
    }
)

response.raise_for_status()

DATABRICKS_TOKEN = response.json()["access_token"]

print("OAuth token generated successfully")

### Create the Managed Memory Store

In [ ]:
import requests

url = f"{workspace_host}/api/2.1/unity-catalog/memory-stores"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

payload = {
    "name": "user_preferences",
    "catalog_name": "YOUR-CATALOG-NAME-GOES-HERE",
    "schema_name": "YOUR-CATALOG-SCHEMA-NAME-GOES-HERE",
    "description": "Long-term memory for storage of user preferences and personal information"
}

response = requests.post(
    url,
    headers=headers,
    json=payload
)

print("Status Code:", response.status_code)
print(response.json())

### Set the Memory Store and Memory Scope Variables

In [ ]:
MEMORY_STORE = "YOUR-CATALOG-NAME.YOUR-SCHEMA-NAME.user_preferences"

# Partition for this demo user's memories
MEMORY_SCOPE = "uid-12345"

print("Memory Store:", MEMORY_STORE)
print("Memory Scope:", MEMORY_SCOPE)

### Upload the Memory Content

In [ ]:
memory_url = (
    f"{workspace_host}/api/2.1/unity-catalog/"
    f"memory-stores/{MEMORY_STORE}/entries"
)

memory_content = """
My name is Kuljot Singh Bakshi.

I am a Udemy Instructor Partner.

My company is CarbonOps.

Our AI product is ESGOps, an AI-powered ESG audit and
reporting tool supporting BRSR, GRI, ESRS, SDGs and
other sustainability reporting frameworks.

Whenever writing LinkedIn posts for me, use a
professional and educational tone.

My preferred audience is sustainability consultants.
"""

payload = {
    "path": "/memories/profile.md",
    "contents": memory_content,
    "description": (
        "User profile containing company, product, "
        "marketing preferences and target audience."
    )
}

response = requests.post(
    memory_url,
    headers=headers,
    params={"scope": MEMORY_SCOPE},
    json=payload
)

print("Status Code:", response.status_code)
print(response.text)

### Define the User Query

In [ ]:
user_question = "Create a short LinkedIn post announcing my AI product."

print("USER:")
print(user_question)

### Retrive the Memory Context

In [ ]:
search_url = (
    f"{workspace_host}/api/2.1/unity-catalog/"
    f"memory-stores/{MEMORY_STORE}/entries:search"
)

payload = {
    "scope": MEMORY_SCOPE,
    "query": """
    Find information relevant to the user's company,
    AI product, LinkedIn writing preferences and
    preferred audience.
    """,
    "top_k": 5
}

response = requests.post(
    search_url,
    headers=headers,
    json=payload
)

print("Status Code:", response.status_code)

response.raise_for_status()

search_results = response.json()

print(search_results)

In [ ]:
results = search_results.get("results", [])

memory_context = "\n\n".join(
    result["memory_entry"]["contents"]
    for result in results
    if result.get("memory_entry", {}).get("contents")
)

print("=== RETRIEVED MEMORY ===")
print(memory_context)

### Send Memory Context with the LLM API Call

In [ ]:
user_prompt = f"""You are a personalized marketing assistant.

The following information was retrieved from the user's
long-term Databricks Managed Memory:

--- MEMORY ---

{memory_context}

--- END MEMORY ---

Use this information whenever relevant to personalize
your response.

Do not mention the memory system or retrieved context. 

The user query is:
{user_question}
"""

In [ ]:
from databricks_openai import DatabricksOpenAI
from databricks.sdk import WorkspaceClient

chat_workspace_client = WorkspaceClient()

client = DatabricksOpenAI(
    workspace_client=chat_workspace_client,
    use_ai_gateway=True
)

response = client.responses.create(
    model="system.ai.claude-sonnet-4-5",
    input=user_prompt,
    stream=False
)

print(response.output_text)